[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day3_lecture.ipynb)

# Day 3 · 강의 — 머신러닝

경사 하강법 · scikit-learn · 검증과 평가 · 회귀

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

수업을 따라가며 진행한다.

**실습** 셀은 그대로 실행해 결과를 눈으로 확인한다.
**문제** 셀은 수업 중에 같이 푼다.

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 학습의 원리

**실습.** 경사 하강법은 기울기의 반대로 조금씩 내려간다. 손으로 한 번 굴려 본다.

In [ ]:
def f(x):      return (x - 3) ** 2 + 1     # 최솟값은 x = 3
def grad(x):   return 2 * (x - 3)

x, lr = 10.0, 0.1
for step in range(1, 21):
    x = x - lr * grad(x)
    if step % 5 == 0:
        print(f"{step:>3}회  x={x:6.3f}  f(x)={f(x):6.3f}")

**실습.** 손실 함수는 예측이 얼마나 틀렸는지를 숫자 하나로 만든다.

In [ ]:
import numpy as np

y_true = np.array([170.0, 175.0, 168.0, 180.0])
y_pred = np.array([172.0, 174.0, 165.0, 179.0])

mse = ((y_true - y_pred) ** 2).mean()
mae = np.abs(y_true - y_pred).mean()
print('MSE', round(mse, 3), ' MAE', round(mae, 3))

# 이상치를 하나 섞으면 MSE 만 크게 뛴다
y_pred2 = y_pred.copy(); y_pred2[0] = 120.0
print('이상치 후 MSE', round(((y_true - y_pred2) ** 2).mean(), 1),
      ' MAE', round(np.abs(y_true - y_pred2).mean(), 1))

> **빈칸 문제 1.** 학습률을 `0.01` 로 낮추고 같은 20회를 돌린 뒤 `x` 를 확인한다.
값이 3에 **덜 가까워지는 것**을 본다.

In [ ]:
def grad(x): return 2 * (x - 3)
x = 10.0
lr = ___
for _ in range(20):
    x = x - lr * grad(x)

assert x > 4, f'학습률이 작으면 20회로는 못 간다. 실제 {x}'
print('통과 — x =', round(x, 3))

> **빈칸 문제 2.** 학습률을 `1.1` 로 올리면 어떻게 되는지 본다. `x` 가 **발산**한다.

In [ ]:
def grad(x): return 2 * (x - 3)
x = 10.0
lr = ___
for _ in range(20):
    x = x - lr * grad(x)

assert abs(x) > 100, f'학습률이 너무 크면 튕겨 나간다. 실제 {x}'
print('통과 — x =', round(x, 1))

## 2. scikit-learn — 네 줄로 끝나는 학습

**실습.** 어떤 모델이든 만들고·학습하고·예측하고·점수 보는 네 줄이다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)

from sklearn.linear_model import LogisticRegression

X_tr, X_te, y_tr, y_te = split()
sc = StandardScaler()
X_tr_s, X_te_s = sc.fit_transform(X_tr), sc.transform(X_te)

model = LogisticRegression(max_iter=1000)   # 만들고
model.fit(X_tr_s, y_tr)                     # 학습하고
pred = model.predict(X_te_s)                # 예측하고
print(round(model.score(X_te_s, y_te), 3))  # 점수 본다

**실습.** 계수를 보면 어떤 열이 답을 밀고 당기는지 드러난다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)
from sklearn.linear_model import LogisticRegression

X_tr, X_te, y_tr, y_te = split()
sc = StandardScaler()
model = LogisticRegression(max_iter=1000).fit(sc.fit_transform(X_tr), y_tr)

coef = dict(zip(X_tr.columns, model.coef_[0].round(2)))
for k, v in sorted(coef.items(), key=lambda kv: -abs(kv[1]))[:5]:
    print(f"{k:>16}  {v:+.2f}")

**실습.** 트리가 무엇을 보고 갈랐는지 글로 뽑아 볼 수 있다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)
from sklearn.tree import DecisionTreeClassifier, export_text

X_tr, X_te, y_tr, y_te = split()
tree = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_tr, y_tr)
print(export_text(tree, feature_names=list(X_tr.columns)))

> **빈칸 문제 4.** 로지스틱 회귀를 학습하고 **테스트 정확도**를 `acc` 에 담는다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)
from sklearn.linear_model import LogisticRegression
X_tr, X_te, y_tr, y_te = split()
sc = StandardScaler()
X_tr_s, X_te_s = sc.fit_transform(X_tr), sc.transform(X_te)
model = ___
model.fit(X_tr_s, y_tr)
acc = ___

assert acc > 0.85, f'0.85 는 넘어야 한다. 실제 {acc}'
print('통과 — 정확도', round(acc, 3))

## 3. 검증과 평가

**실습.** 정확도만 보면 속는다. 전부 양품이라 찍어도 81% 가 나온다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)

X_tr, X_te, y_tr, y_te = split()
print('테스트셋 양품 비율:', round(y_te.mean(), 3))
print('전부 1이라 찍은 정확도:', round((y_te == 1).mean(), 3))

**실습.** 혼동 행렬은 어디서 틀렸는지 네 칸으로 보여 준다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

X_tr, X_te, y_tr, y_te = split()
model = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_tr, y_tr)
pred = model.predict(X_te)

print(confusion_matrix(y_te, pred))
print(classification_report(y_te, pred, target_names=['불량', '양품']))

> **빈칸 문제 7.** **전부 양품이라 찍는** 예측을 만들어 정확도를 `dumb_acc` 에 담는다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)
import numpy as np
X_tr, X_te, y_tr, y_te = split()
pred = ___
dumb_acc = ___

assert abs(dumb_acc - 0.81) < 0.02, f'실제 {dumb_acc}'
print('통과 — 아무것도 안 배워도', round(dumb_acc, 3))

## 4. 회귀 — 용량 맞히기

**실습.** 분류가 아니라 숫자를 맞힌다. 정답 열만 바꾸면 나머지는 같다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

X_tr, X_te, y_tr, y_te = split(target='방전용량')

lin = LinearRegression().fit(X_tr, y_tr)
rf = RandomForestRegressor(n_estimators=200, random_state=42).fit(X_tr, y_tr)

for name, m in (('linear', lin), ('forest', rf)):
    p = m.predict(X_te)
    print(f"{name:>8}  R2 {r2_score(y_te, p):.3f}  RMSE {mean_squared_error(y_te, p) ** 0.5:.2f}")